# qust 参数与策略参数优化：从 0 到完整曲线对比

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


本节按真实研究流程说明 qust 参数系统和参数优化，不只给一段 `optuna_params` 代码：

1. 先理解什么是 `Params`，为什么策略参数不要写死；
2. 用 `pms(...)` 定义一个带参数的均线策略；
3. 先跑默认参数，得到默认净值曲线；
4. 用 `opt_params` 做完整网格搜索；
5. 把网格搜索里最好的参数写回新的策略表达式；
6. 用 `optuna_params` 做启发式搜索，并说明它返回的是新的 Expr；
7. 对比默认参数、网格最优、Optuna 最优的收益曲线；
8. 说明每种优化方法适合什么场景。

重要：这里的图使用 qust 原生 `monitor` 输出，不转成 PNG。运行到 plot cell 时，看到的就是 qust 实际 monitor。


In [1]:
import optuna

import qust as qs

from qust import col, pms
from qust._polars import pl

optuna.logging.set_verbosity(optuna.logging.WARNING)
pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(14)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"


## 1. 准备数据：先用一个品种的小样本

参数优化本质上是“重复跑很多次策略”。如果一开始就用全市场全历史，调试会很慢，也不容易看清每一步。

所以这里先取一个品种 `au` 的前 2000 行 K 线。真实研究时只需要把 `head(2_000)` 去掉或换成更大的范围。


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])
PLOT_ROW = raw.select("ticker", "ct").unique().sort(["ticker", "ct"]).row(0, named=True)
PLOT_TICKER = PLOT_ROW["ticker"]
PLOT_CT = PLOT_ROW["ct"]
data = raw.filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == PLOT_CT)).head(2_000)

print("raw shape:", raw.shape)
print("optimization contract:", PLOT_TICKER, PLOT_CT)
print("optimization sample shape:", data.shape)
data.select("ticker", "ct", "datetime", "open", "high", "low", "close", "volume").head(8)


raw shape: (610463, 8)
optimization sample shape: (2000, 8)


ticker,datetime,open,high,low,close,volume
str,datetime[ms],f64,f64,f64,f64,f64
"""au""",2022-07-02 00:01:00,390.160004,390.160004,390.059998,390.119995,81.0
"""au""",2022-07-02 00:02:00.500,390.119995,390.140015,390.079987,390.140015,52.0
"""au""",2022-07-02 00:03:00,390.140015,390.200012,390.119995,390.200012,50.0
"""au""",2022-07-02 00:04:01,390.200012,390.220001,390.140015,390.160004,61.0
"""au""",2022-07-02 00:05:00.500,390.140015,390.140015,390.079987,390.100006,41.0
"""au""",2022-07-02 00:06:00.500,390.079987,390.200012,390.079987,390.140015,73.0
"""au""",2022-07-02 00:07:00,390.119995,390.140015,389.859985,389.880005,172.0
"""au""",2022-07-02 00:08:03,389.859985,389.940002,389.859985,389.880005,118.0


## 2. `Params` 是什么？

普通写法会把策略参数写死：

```python
fast = 10
slow = 60
fee = 0.0002
```

这样能跑，但优化器不知道哪些数字应该搜索。qust 用 `pms(...)` 创建参数对象：

```python
pms(5, 30).title("fast").value(10).step(5)
```

含义是：

- 搜索空间从 5 到 30；
- 参数名是 `fast`，后面 `include=["fast"]` 和 Optuna 都靠这个名字识别；
- 默认值是 10；
- 搜索步长是 5。

参数类型：

- `pms(1, 20)`：整数参数；
- `pms(0.0, 1.0)`：浮点参数；
- `pms(["a", "b"], None)`：字符串/类别参数；
- `pms([expr1, expr2], None)`：表达式候选池，本质上也是类别参数。


In [3]:
def param_table(expr):
    rows = []
    for idx, param in enumerate(expr.params()):
        widget = param.widget()
        rows.append({
            "index": idx,
            "name": None if widget is None else widget.get("title"),
            "kind": None if widget is None else widget.get("kind"),
            "value": None if widget is None else widget.get("value"),
            "start": None if widget is None else widget.get("start"),
            "end": None if widget is None else widget.get("end"),
            "step": None if widget is None else widget.get("step"),
            "pool": None if widget is None else widget.get("pool"),
        })
    return pl.DataFrame(rows)

fast_demo = pms(5, 30).title("fast").value(10).step(5)
fee_demo = pms(0.0, 0.001).title("fee").value(0.0002).step(0.0002)
mode_demo = pms(["long_only", "flat"], None).title("mode").value("long_only")

demo_expr = col(
    fast_demo.as_expr().alias("fast_value"),
    fee_demo.as_expr().alias("fee_value"),
    mode_demo.as_expr().alias("mode_value"),
)
param_table(demo_expr)


index,name,kind,value,start,end,step,pool
i64,str,str,str,f64,f64,f64,list[str]
0,"""fast""","""usize""","""10""",5.0,30.0,5.0,null
1,"""fee""","""f64""","""0.0002""",0.0,0.001,0.0002,null
2,"""mode""","""string""","""long_only""",null,null,null,"[""long_only"", ""flat""]"


## 3. 定义一个带参数的策略

策略逻辑非常简单，目的是讲清参数优化流程：

1. 计算快均线 `fast_ma`；
2. 计算慢均线 `slow_ma`；
3. 如果 `fast_ma > slow_ma`，下一根 K 线持有多头；否则空仓；
4. 每次仓位变化时扣手续费 `fee`；
5. 输出逐行收益 `ret` 和累计净值 `equity`。

这里有三个可优化参数：

- `fast`：快均线窗口；
- `slow`：慢均线窗口；
- `fee`：交易成本。

注意 `hold` 用了 `shift(1).expanding()`，表示今天算出的信号下一根 bar 才生效，避免用当前 bar 收盘信号交易当前 bar 的收益。


In [4]:
def make_ma_strategy_curve_expr():
    fast = pms(5, 30).title("fast").value(10).step(5)
    slow = pms(40, 100).title("slow").value(60).step(20)
    fee = pms(0.0, 0.001).title("fee").value(0.0002).step(0.0002)

    return (
        col
        .with_cols(
            col("close").mean().rolling(fast).alias("fast_ma"),
            col("close").mean().rolling(slow).alias("slow_ma"),
        )
        .with_cols(
            (col("fast_ma") > col("slow_ma"))
            .cast(pl.Float64)
            .fill_null(col.lit(0.0))
            .alias("hold_raw")
        )
        .with_cols(
            col("hold_raw")
            .shift(1)
            .expanding()
            .fill_null(col.lit(0.0))
            .alias("hold")
        )
        .with_cols(
            (
                col("close").pct().expanding().fill_null(col.lit(0.0)) * col("hold")
                - (col("hold") - col("hold").shift(1).expanding().fill_null(col.lit(0.0))).abs()
                * fee.as_expr()
            ).alias("ret")
        )
        .with_cols(
            (col("ret").sum().expanding() + col.lit(1.0)).alias("equity")
        )
        .select("datetime", "close", "fast_ma", "slow_ma", "hold", "ret", "equity")
    )

def make_ma_strategy_score_expr():
    curve = make_ma_strategy_curve_expr()
    return curve.select(
        (col("ret").mean() / col("ret").std() * col.lit(252 ** 0.5)).alias("score")
    )

curve_expr = make_ma_strategy_curve_expr()
score_expr = make_ma_strategy_score_expr()

print(curve_expr.display()[:900] + "...")
param_table(curve_expr)


col.all.with_cols(col("close").mean().rolling(10, 10).alias("fast_ma"), col("close").mean().rolling(60, 60).alias("slow_ma")).with_cols(col("fast_ma") > col("slow_ma").cast(pl.Float64).fill_null(col.lit(0.0)).alias("hold_raw")).with_cols(col("hold_raw").shift(1).expanding().fill_null(col.lit(0.0)).alias("hold")).with_cols(((col("close") / col("close").shift(1)) - col.lit(1.0).expanding().fill_null(col.lit(0.0)) * col("hold")) - (col("hold") - col("hold").shift(1).expanding().fill_null(col.lit(0.0)).abs() * pms(...).title("fee").value(0.0002)).alias("ret")).with_cols(col("ret").sum().expanding() + col.lit(1.0).alias("equity")).select(col("datetime"), col("close"), col("fast_ma"), col("slow_ma"), col("hold"), col("ret"), col("equity"))...


index,name,kind,value,start,end,step,pool
i64,str,str,f64,f64,f64,f64,null
0,"""fast""","""usize""",10.0,5.0,30.0,5.0,null
1,"""slow""","""usize""",60.0,40.0,100.0,20.0,null
2,"""fee""","""f64""",0.0002,0.0,0.001,0.0002,null


## 4. 默认参数先跑一遍

优化之前必须先知道默认参数表现，否则“优化后好不好”没有参照物。

下面输出：

- 默认参数表；
- 默认参数最终几行，包括净值 `equity`；
- 默认参数的评分 `score`。


In [5]:
default_curve = curve_expr.calc_data(data)
default_score = score_expr.calc_data(data)

display(param_table(curve_expr))
display(default_curve.tail(8))
display(default_score)


index,name,kind,value,start,end,step,pool
i64,str,str,f64,f64,f64,f64,null
0,"""fast""","""usize""",10.0,5.0,30.0,5.0,null
1,"""slow""","""usize""",60.0,40.0,100.0,20.0,null
2,"""fee""","""f64""",0.0002,0.0,0.001,0.0002,null


datetime,close,fast_ma,slow_ma,hold,ret,equity
datetime[ms],f64,f64,f64,f64,f64,f64
2022-07-07 14:28:01,377.660004,377.582001,377.597667,0.0,-0.0,0.994648
2022-07-07 14:29:02,377.579987,377.582001,377.596333,0.0,-0.0,0.994648
2022-07-07 14:30:01,377.619995,377.589999,377.597333,0.0,0.0,0.994648
2022-07-07 14:31:01,377.579987,377.585999,377.599333,0.0,-0.0,0.994648
2022-07-07 14:32:00.500,377.440002,377.575998,377.600333,0.0,-0.0,0.994648
2022-07-07 14:33:04.500,377.440002,377.567999,377.598333,0.0,0.0,0.994648
2022-07-07 14:34:03,377.440002,377.566,377.595333,0.0,0.0,0.994648
2022-07-07 14:35:00,377.5,377.556,377.592667,0.0,0.0,0.994648


score
f64
-0.310535


## 5. 用 qust 画默认参数净值曲线

输入给 monitor 的是两列：`datetime, equity`。

这张图是后面优化前/后的基准线。


In [6]:
default_curve_plot = col("datetime", "equity").monitor("default_equity", show_axis_label=True).line().runtime()
default_curve_plot.plot(default_curve, open_in_jupyter=True, auto_open=False, height=460)


## 6. `opt_params`：完整网格搜索

`opt_params` 会遍历参数空间里的所有组合。这里三个参数组合数量是：

- fast: 5, 10, 15, 20, 25, 30，共 6 个；
- slow: 40, 60, 80, 100，共 4 个；
- fee: 0.0 到 0.001，步长 0.0002，共 6 个；
- 总计 6 * 4 * 6 = 144 次回测。

优点：结果完整，能看到所有组合。

缺点：参数一多组合数会爆炸。

`parallel=False` 表示单核跑，最稳、最省内存；如果表达式中间结果不大，可以改成 `parallel=True` 或指定整数线程数。


In [7]:
grid = (
    score_expr
    .opt_params(include=["fast", "slow", "fee"], parallel=False)
    .runtime()
    .calc_data(data)
    .sort("score", descending=True)
)

grid.head(12)


score,fast,slow,fee
f64,u32,u32,f64
0.290947,15,40,0.0
0.260445,20,40,0.0
0.25435,15,60,0.0
0.158382,10,60,0.0
0.124687,10,40,0.0
0.097922,5,80,0.0
0.093554,10,80,0.0
0.082929,5,60,0.0
0.033894,25,40,0.0


## 7. 把网格最优参数写回新的策略表达式

注意不要在原表达式上乱改。这里重新创建一份策略表达式，然后把 `grid` 里第一行的最优参数写进去。

这一步等价于：

1. `best = grid[0]`；
2. 新建一个策略表达式；
3. 找到同名参数；
4. `set_value_inplace(best_value)`；
5. 再跑一次完整净值曲线。


In [8]:
def apply_param_values(expr, values: dict):
    for param in expr.params():
        widget = param.widget()
        if widget is None:
            continue
        name = widget.get("title")
        if name in values:
            param.set_value_inplace(values[name])
    return expr

best_grid_values = {
    "fast": int(grid["fast"][0]),
    "slow": int(grid["slow"][0]),
    "fee": float(grid["fee"][0]),
}

grid_best_curve_expr = apply_param_values(make_ma_strategy_curve_expr(), best_grid_values)
grid_best_score_expr = apply_param_values(make_ma_strategy_score_expr(), best_grid_values)

grid_best_curve = grid_best_curve_expr.calc_data(data)
grid_best_score = grid_best_score_expr.calc_data(data)

print("best_grid_values =", best_grid_values)
display(param_table(grid_best_curve_expr))
display(grid_best_score)
display(grid_best_curve.tail(8))


best_grid_values = {'fast': 15, 'slow': 40, 'fee': 0.0}


index,name,kind,value,start,end,step,pool
i64,str,str,f64,f64,f64,f64,null
0,"""fast""","""usize""",15.0,5.0,30.0,5.0,null
1,"""slow""","""usize""",40.0,40.0,100.0,20.0,null
2,"""fee""","""f64""",0.0,0.0,0.001,0.0002,null


score
f64
0.290947


datetime,close,fast_ma,slow_ma,hold,ret,equity
datetime[ms],f64,f64,f64,f64,f64,f64
2022-07-07 14:28:01,377.660004,377.594668,377.5585,1.0,-0.000106,1.005415
2022-07-07 14:29:02,377.579987,377.596,377.556,1.0,-0.000212,1.005203
2022-07-07 14:30:01,377.619995,377.593333,377.553,1.0,0.000106,1.005309
2022-07-07 14:31:01,377.579987,377.586666,377.548499,1.0,-0.000106,1.005203
2022-07-07 14:32:00.500,377.440002,377.573332,377.541,1.0,-0.000371,1.004832
2022-07-07 14:33:04.500,377.440002,377.565332,377.532,1.0,0.0,1.004832
2022-07-07 14:34:03,377.440002,377.556,377.524,1.0,0.0,1.004832
2022-07-07 14:35:00,377.5,377.553333,377.5195,1.0,0.000159,1.004991


## 8. `optuna_params`：启发式搜索

`optuna_params` 不遍历全部组合，而是让 Optuna 根据历史 trial 继续建议参数。

关键点：

- `score_fn=None` 时，默认取结果第一列第一行作为目标值，所以我们的 `score_expr` 可以直接优化；
- `optuna_params` 返回的是新的 Expr，不会修改原来的 `score_expr`；
- 当前实现会把默认参数作为 warm-start trial，所以默认参数本身也会被纳入比较。

下面用 40 次 trial 搜索。为了演示可重复，指定一个 fixed seed 的 sampler。


In [9]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=7),
)

optuna_best_score_expr = score_expr.optuna_params(
    data,
    n_trials=40,
    study=study,
    show_progress_bar=False,
)

trial_rows = []
for trial in study.trials:
    row = {"trial": trial.number, "value": trial.value}
    row.update({f"param_{k}": v for k, v in trial.params.items()})
    trial_rows.append(row)

trials = pl.DataFrame(trial_rows).sort("value", descending=True)

display(param_table(score_expr).with_columns(pl.lit("original score expr").alias("expr")))
display(param_table(optuna_best_score_expr).with_columns(pl.lit("optuna best score expr").alias("expr")))
display(trials.head(12))


[W 2026-07-29 18:12:42,550] Trial 1 failed with parameters: {'fast': 5, 'slow': 60, 'fee': 0.0002} because of the following error: The value nan is not acceptable.


[W 2026-07-29 18:12:42,551] Trial 1 failed with value nan.


[W 2026-07-29 18:12:42,569] Trial 4 failed with parameters: {'fast': 10, 'slow': 40, 'fee': 0.0002} because of the following error: The value nan is not acceptable.


[W 2026-07-29 18:12:42,569] Trial 4 failed with value nan.


[W 2026-07-29 18:12:42,608] Trial 10 failed with parameters: {'fast': 5, 'slow': 100, 'fee': 0.0004} because of the following error: The value nan is not acceptable.


...

[W 2026-07-29 18:12:42,875] Trial 39 failed with parameters: {'fast': 30, 'slow': 40, 'fee': 0.001} because of the following error: The value nan is not acceptable.


[W 2026-07-29 18:12:42,876] Trial 39 failed with value nan.


index,name,kind,value,start,end,step,pool,expr
i64,str,str,f64,f64,f64,f64,null,str
0,"""fast""","""usize""",10.0,5.0,30.0,5.0,null,"""original score expr"""
1,"""slow""","""usize""",60.0,40.0,100.0,20.0,null,"""original score expr"""
2,"""fee""","""f64""",0.0002,0.0,0.001,0.0002,null,"""original score expr"""


index,name,kind,value,start,end,step,pool,expr
i64,str,str,f64,f64,f64,f64,null,str
0,"""fast""","""usize""",10.0,5.0,30.0,5.0,null,"""optuna best score expr"""
1,"""slow""","""usize""",60.0,40.0,100.0,20.0,null,"""optuna best score expr"""
2,"""fee""","""f64""",0.0,0.0,0.001,0.0002,null,"""optuna best score expr"""


trial,value,param_fast,param_slow,param_fee
i64,f64,i64,i64,f64
1,null,5,60,0.0002
4,null,10,40,0.0002
10,null,5,100,0.0004
12,null,20,40,0.0002
14,null,30,40,0.001
15,null,30,40,0.001
16,null,30,40,0.001
17,null,30,40,0.001
18,null,30,40,0.001


## 9. 用 Optuna 最优参数生成策略曲线

`optuna_params` 返回的是“优化后的 score 表达式”。为了画策略曲线，我们把它的参数值读出来，再写到一份新的 curve 表达式里。


In [10]:
optuna_values = {}
for param in optuna_best_score_expr.params():
    widget = param.widget()
    if widget is not None:
        optuna_values[widget["title"]] = widget["value"]

optuna_best_curve_expr = apply_param_values(make_ma_strategy_curve_expr(), optuna_values)
optuna_best_curve = optuna_best_curve_expr.calc_data(data)
optuna_best_score = optuna_best_score_expr.calc_data(data)

print("optuna_values =", optuna_values)
display(optuna_best_score)
display(optuna_best_curve.tail(8))


optuna_values = {'fast': 10, 'slow': 60, 'fee': 0.0}


score
f64
0.158382


datetime,close,fast_ma,slow_ma,hold,ret,equity
datetime[ms],f64,f64,f64,f64,f64,f64
2022-07-07 14:28:01,377.660004,377.582001,377.597667,0.0,-0.0,1.002648
2022-07-07 14:29:02,377.579987,377.582001,377.596333,0.0,-0.0,1.002648
2022-07-07 14:30:01,377.619995,377.589999,377.597333,0.0,0.0,1.002648
2022-07-07 14:31:01,377.579987,377.585999,377.599333,0.0,-0.0,1.002648
2022-07-07 14:32:00.500,377.440002,377.575998,377.600333,0.0,-0.0,1.002648
2022-07-07 14:33:04.500,377.440002,377.567999,377.598333,0.0,0.0,1.002648
2022-07-07 14:34:03,377.440002,377.566,377.595333,0.0,0.0,1.002648
2022-07-07 14:35:00,377.5,377.556,377.592667,0.0,0.0,1.002648


## 10. 优化前 vs 网格最优 vs Optuna 最优：净值曲线对比

现在把三条曲线拼到同一张表：

- `default_equity`：默认参数；
- `grid_best_equity`：网格搜索最优参数；
- `optuna_best_equity`：Optuna 搜索最优参数。

这一步最重要：优化不能只看参数表，必须看收益曲线是否合理。比如曲线是否来自少数几笔极端收益、是否回撤过大、是否只是手续费设成 0 后变好。


In [11]:
compare_curve = default_curve.select(
    "datetime",
    pl.col("equity").alias("default_equity"),
).with_columns(
    grid_best_curve["equity"].alias("grid_best_equity"),
    optuna_best_curve["equity"].alias("optuna_best_equity"),
)

compare_curve.tail(8)


datetime,default_equity,grid_best_equity,optuna_best_equity
datetime[ms],f64,f64,f64
2022-07-07 14:28:01,0.994648,1.005415,1.002648
2022-07-07 14:29:02,0.994648,1.005203,1.002648
2022-07-07 14:30:01,0.994648,1.005309,1.002648
2022-07-07 14:31:01,0.994648,1.005203,1.002648
2022-07-07 14:32:00.500,0.994648,1.004832,1.002648
2022-07-07 14:33:04.500,0.994648,1.004832,1.002648
2022-07-07 14:34:03,0.994648,1.004832,1.002648
2022-07-07 14:35:00,0.994648,1.004991,1.002648


In [12]:
compare_plot = col(
    col("datetime", "default_equity", "grid_best_equity", "optuna_best_equity")
        .monitor("equity_compare", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([["equity_compare"]]).runtime()

compare_plot.plot(compare_curve, open_in_jupyter=True, auto_open=False, height=560)


## 11. 对比统计表

曲线看形状，统计表看数字。这里用 qust 表达式从每条曲线里提取最后净值和最大回撤。


In [13]:
def summarize_curve(df, name, equity_col):
    return col(
        col.lit(name).alias("name"),
        col(equity_col).last_value().alias("final_equity"),
        (col(equity_col).last_value() - col.lit(1.0)).alias("total_return"),
        col(equity_col).bt.drawdown().min().alias("max_drawdown"),
    ).calc_data(df)

summary = pl.concat([
    summarize_curve(compare_curve, "default", "default_equity"),
    summarize_curve(compare_curve, "grid_best", "grid_best_equity"),
    summarize_curve(compare_curve, "optuna_best", "optuna_best_equity"),
])

summary


name,final_equity,total_return,max_drawdown
str,f64,f64,f64
"""default""",0.994648,-0.005352,-0.010914
"""grid_best""",1.004991,0.004991,-0.003954
"""optuna_best""",1.002648,0.002648,-0.005082


## 12. 选择哪种优化方法？

| 方法 | 适合场景 | 优点 | 风险 |
| --- | --- | --- | --- |
| 手动 `.value(...)` | 先跑通策略、复现实验 | 最可控 | 不能系统搜索 |
| `opt_params` | 参数少、离散网格、想看完整平面 | 不漏组合，结果可解释 | 组合数爆炸，内存和时间压力大 |
| `optuna_params` | 参数多、连续空间、先找方向 | trial 少时也能探索 | 不保证找到全局最优，需要设置 seed/study 保存记录 |
| `include=[...]` | 只优化部分参数 | 避免把所有参数同时放开 | 忽略参数交互 |
| 自定义 `score_fn` | 多指标/风险惩罚 | 可以把回撤、交易次数等纳入目标 | score 写错会优化错方向 |

实盘或严肃研究里还应该做：

1. 训练集/验证集拆分；
2. 交易成本和滑点敏感性测试；
3. 参数稳定性分析，不只看最优点；
4. 避免只优化夏普，不看交易次数和容量；
5. 用更长样本和更多品种复核。
